<a href="https://colab.research.google.com/github/naimul-islam-64/Backup/blob/main/01_tool_calling_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔧 Notebook 1 — Tool Calling Fine-Tuning
**Model:** `Qwen2.5-0.5B-Instruct`  
**Dataset:** `NousResearch/hermes-function-calling-v1`  
**Method:** QLoRA (4-bit) via Unsloth — Lightning fast ⚡

### 📋 Workflow
1. Install Unsloth + dependencies  
2. Load Qwen2.5-0.5B with 4-bit QLoRA  
3. Load & preprocess hermes dataset  
4. Train with SFTTrainer  
5. Test the model  
6. Save / push to HuggingFace Hub  

> ⏱️ **Expected training time on free Colab T4:** ~20–35 minutes

## 🔷 Step 1 — Install Dependencies

In [ ]:
# Install Unsloth (handles transformers, peft, trl, xformers automatically)
# This takes ~2-3 minutes on first run
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
print("✅ Installation complete!")

## 🔷 Step 2 — Load Model with QLoRA

In [2]:
from unsloth import FastLanguageModel
import torch

# ─── CONFIG ────────────────────────────────────────────────────────────────
MODEL_NAME     = "unsloth/Qwen2.5-0.5B-Instruct"  # Unsloth's optimized version
MAX_SEQ_LENGTH = 2048   # Covers most tool-call examples. Reduce to 1024 for speed.
DTYPE          = None   # Auto-detect: float16 on T4, bfloat16 on A100/H100
LOAD_IN_4BIT   = True   # QLoRA — saves ~70% VRAM
# ────────────────────────────────────────────────────────────────────────────

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)
print(f"✅ Base model loaded: {MODEL_NAME}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Base model loaded: unsloth/Qwen2.5-0.5B-Instruct


In [3]:
# Attach LoRA adapters — only these layers get trained (fast + memory-efficient)
model = FastLanguageModel.get_peft_model(
    model,
    r               = 16,       # LoRA rank. 16 is the sweet spot for quality vs speed.
    target_modules  = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha      = 32,       # Scaling factor. Typically 2x rank.
    lora_dropout    = 0,        # 0 = fastest. Non-zero adds regularization.
    bias            = "none",
    use_gradient_checkpointing = "unsloth",  # Saves extra VRAM, Unsloth's custom impl
    random_state    = 42,
    use_rslora      = False,
    loftq_config    = None,
)

# Show trainable parameter count
model.print_trainable_parameters()

Unsloth 2026.3.4 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## 🔷 Step 3 — Load & Preprocess Dataset

The Hermes dataset uses **ShareGPT format**: each row has a `conversations` list with `{"from": ..., "value": ...}` turns.  
Roles: `system` → system, `human` → user, `gpt` → assistant.  
We convert these to Qwen's **ChatML format** (`<|im_start|>` / `<|im_end|>`).

In [5]:
from datasets import load_dataset, concatenate_datasets

# Load all 4 subsets of the hermes tool-calling dataset
print("Loading hermes-function-calling-v1 dataset...")

subsets = [
    "func_calling",
    "json_mode_agentic",
    "json_mode_singleturn",
    "glaive_func_calling",        # ← fixed
    "func_calling_singleturn",    # ← bonus: grab this one too
]

datasets_list = []
for subset in subsets:
    try:
        ds = load_dataset(
            "NousResearch/hermes-function-calling-v1",
            name   = subset,
            split  = "train",
        )
        # Keep only the columns we need
        if "conversations" in ds.column_names:
            datasets_list.append(ds.select_columns(["conversations"]))
            print(f"  ✅ {subset}: {len(ds):,} examples")
        else:
            print(f"  ⚠️  {subset}: no 'conversations' column, skipping")
    except Exception as e:
        print(f"  ⚠️  {subset}: failed ({e}), skipping")

raw_dataset = concatenate_datasets(datasets_list)
print(f"\n📊 Total examples loaded: {len(raw_dataset):,}")

Loading hermes-function-calling-v1 dataset...
  ✅ func_calling: 1,893 examples
  ✅ json_mode_agentic: 1,342 examples
  ✅ json_mode_singleturn: 1,241 examples


glaive-function-calling-5k.json:   0%|          | 0.00/20.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5209 [00:00<?, ? examples/s]

  ✅ glaive_func_calling: 5,209 examples


func-calling-singleturn.json:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1893 [00:00<?, ? examples/s]

  ✅ func_calling_singleturn: 1,893 examples

📊 Total examples loaded: 11,578


In [6]:
# Inspect a raw sample so we understand the format
sample = raw_dataset[0]
print("Raw sample structure:")
for i, turn in enumerate(sample["conversations"][:3]):
    print(f"  Turn {i}: from='{turn['from']}', value[:80]='{str(turn['value'])[:80]}...'")

Raw sample structure:
  Turn 0: from='system', value[:80]='You are a function calling AI model. You are provided with function signatures w...'
  Turn 1: from='human', value[:80]='I've recently installed a new security system at my home, and I want to ensure e...'
  Turn 2: from='gpt', value[:80]='<tool_call>
{"name": "get_camera_live_feed", "arguments": {"camera_id": "front_d...'


In [7]:
# ──────────────────────────────────────────────────────────────────────────
# Convert ShareGPT format → Qwen ChatML format
# Qwen2.5 uses ChatML: <|im_start|>role\ncontent<|im_end|>\n
# ──────────────────────────────────────────────────────────────────────────

# Role mapping: ShareGPT → ChatML roles
ROLE_MAP = {
    "system"    : "system",
    "human"     : "user",
    "gpt"       : "assistant",
    "tool"      : "tool",
    "function"  : "tool",
}

EOS_TOKEN = tokenizer.eos_token  # <|im_end|> for Qwen

def sharegpt_to_chatml(example):
    """Convert a ShareGPT conversations list into a single ChatML string."""
    convs = example.get("conversations", [])
    if not convs:
        return {"text": ""}

    parts = []
    for turn in convs:
        role  = ROLE_MAP.get(turn.get("from", "").lower(), "user")
        value = turn.get("value", "").strip()
        parts.append(f"<|im_start|>{role}\n{value}<|im_end|>\n")

    # Append EOS token at the end so the model learns when to stop
    text = "".join(parts) + EOS_TOKEN
    return {"text": text}


# Apply the conversion (fast batched map)
dataset = raw_dataset.map(
    sharegpt_to_chatml,
    remove_columns = raw_dataset.column_names,
    num_proc       = 2,
    desc           = "Converting to ChatML",
)

# Filter out empty or very short examples
dataset = dataset.filter(lambda x: len(x["text"]) > 50)

print(f"\n✅ Formatted dataset: {len(dataset):,} examples")
print("\nSample ChatML output:")
print(dataset[0]["text"][:400], "...")

Converting to ChatML (num_proc=2):   0%|          | 0/11578 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11578 [00:00<?, ? examples/s]


✅ Formatted dataset: 11,578 examples

Sample ChatML output:
<|im_start|>system
You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug into functions.
<tools>
[{"type": "function", "function": {"name": "get_camera_live_feed", "description": "Retrieves the live feed from a specified secur ...


In [8]:
# ─── OPTIONAL: Limit dataset size for faster training ────────────────────
# Free Colab T4 has ~12h session limit. 3000 samples ≈ 15-20 min training.
# Set MAX_SAMPLES = None to train on the full dataset (~30-40 min).
MAX_SAMPLES = 3000

if MAX_SAMPLES and MAX_SAMPLES < len(dataset):
    dataset = dataset.shuffle(seed=42).select(range(MAX_SAMPLES))
    print(f"📉 Subsampled to {MAX_SAMPLES:,} examples for speed")
else:
    print(f"📊 Using full dataset: {len(dataset):,} examples")

📉 Subsampled to 3,000 examples for speed


## 🔷 Step 4 — Configure & Run Training

In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = dataset,
    dataset_text_field = "text",
    max_seq_length  = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing         = True,   # 🚀 Packs multiple short samples → 3x speedup!
    args = TrainingArguments(
        # ── Batch & Gradient ──────────────────────────────────────────────
        per_device_train_batch_size    = 2,
        gradient_accumulation_steps    = 4,   # Effective batch = 2 × 4 = 8
        # ── LR Schedule ───────────────────────────────────────────────────
        learning_rate                  = 2e-4,
        lr_scheduler_type              = "cosine",
        warmup_ratio                   = 0.05,
        # ── Duration ──────────────────────────────────────────────────────
        num_train_epochs               = 2,    # 2 epochs is enough for LoRA
        # ── Precision ─────────────────────────────────────────────────────
        fp16                           = not is_bfloat16_supported(),
        bf16                           = is_bfloat16_supported(),
        # ── Optimizer ─────────────────────────────────────────────────────
        optim                          = "adamw_8bit",  # 8-bit Adam saves VRAM
        weight_decay                   = 0.01,
        max_grad_norm                  = 1.0,
        # ── Logging & Saving ──────────────────────────────────────────────
        logging_steps                  = 25,
        save_strategy                  = "epoch",
        output_dir                     = "./tool_calling_checkpoints",
        # ── Misc ──────────────────────────────────────────────────────────
        seed                           = 42,
        report_to                      = "none",  # Disable wandb
        dataloader_num_workers         = 0,       # Stable on Colab
    ),
)

print("✅ Trainer configured!")
print(f"   Packing: True | Batch: {2} × {4} accumulation = {8}")
print(f"   Epochs: 2 | LR: 2e-4 | Optimizer: adamw_8bit")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3000 [00:00<?, ? examples/s]

✅ Trainer configured!
   Packing: True | Batch: 2 × 4 accumulation = 8
   Epochs: 2 | LR: 2e-4 | Optimizer: adamw_8bit


In [10]:
# Show GPU memory before training
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Max VRAM: {max_memory} GB | Reserved before training: {start_gpu_memory} GB")

# 🚀 START TRAINING!
print("\n🏋️ Starting training...")
trainer_stats = trainer.train()

# Report memory usage after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n✅ Training complete!")
print(f"   Total training time: {trainer_stats.metrics['train_runtime']:.1f} seconds")
print(f"   Peak VRAM used: {used_memory} GB / {max_memory} GB")
print(f"   Final loss: {trainer_stats.metrics['train_loss']:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


GPU: Tesla T4
Max VRAM: 14.563 GB | Reserved before training: 0.588 GB

🏋️ Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 2 | Total steps = 750
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
25,1.484799
50,0.862275
75,0.812665
100,0.784570
125,0.688090
150,0.754668
175,0.755021
200,0.763077
225,0.656710
250,0.655901


Step,Training Loss
25,1.484799
50,0.862275
75,0.812665
100,0.784570
125,0.688090
150,0.754668
175,0.755021
200,0.763077
225,0.656710
250,0.655901



✅ Training complete!
   Total training time: 1793.1 seconds
   Peak VRAM used: 6.498 GB / 14.563 GB
   Final loss: 0.5833


## 🔷 Step 5 — Test the Fine-Tuned Model

In [12]:
# Enable fast inference mode
FastLanguageModel.for_inference(model)

# Test: Ask the model to call a weather tool
tools_description = """[{"name": "get_weather", "description": "Get current weather for a city", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "City name"}, "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, "required": ["location"]}}]"""

test_messages = [
    {
        "role": "system",
        "content": f"You are a function calling AI model. You are provided with function signatures within <tools></tools> tags.\n<tools>\n{tools_description}\n</tools>\nFor each function call return a json object with function name and arguments wrapped in <tool_call></tool_call> tags."
    },
    {
        "role": "user",
        "content": "What's the weather like in Bangladesh Dhaka right now?"
    }
]

# Apply Qwen's chat template
input_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize         = False,
    add_generation_prompt = True,
)

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens   = 200,
        temperature      = 0.1,
        do_sample        = True,
        pad_token_id     = tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("📤 Model response:")
print(response)

📤 Model response:
<tool_call>
{"name": "get_weather", "arguments": {"location": "Dhaka", "unit": "fahrenheit"}}
</tool_call>


In [15]:
# Let's see the raw output clearly
test_input = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": "What's the weather in Dhaka in celsius?"},
]
inp = tokenizer.apply_chat_template(test_input, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(inp, return_tensors="pt").to("cuda")

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
print("RAW OUTPUT:")
print(repr(raw))
print("\nFORMATTED:")
print(raw)

RAW OUTPUT:
'<tool_call>{json}\n{"data": {"temperature": 25}}<|im_end|>'

FORMATTED:
<tool_call>{json}
{"data": {"temperature": 25}}<|im_end|>


In [ ]:
import json, re, torch

# ── Much stricter system prompt ───────────────────────────────────────────
STRICT_SYSTEM = (
    "You are a function calling AI. When the user asks about weather, "
    "you MUST call the get_weather tool.\n\n"
    "STRICT OUTPUT FORMAT — no extra text, no markdown, no placeholder data:\n"
    "<tool_call>\n"
    "{\"name\": \"get_weather\", \"arguments\": {\"location\": \"<city>\", \"unit\": \"celsius\"}}\n"
    "</tool_call>\n\n"
    f"Available tool:\n<tools>\n{TOOLS_SCHEMA}\n</tools>\n\n"
    "RULES:\n"
    "1. Output ONLY the <tool_call> block above — nothing else on your first response.\n"
    "2. Do NOT invent weather data. NEVER put numbers or results inside <tool_call>.\n"
    "3. The arguments must only contain 'location' (string) and optionally 'unit'.\n"
    "4. After receiving <tool_result>, answer naturally in plain text."
)

def robust_parse_tool_call(text: str):
    """
    Handles multiple malformed formats the model might output:
    - <tool_call>{"name": ...}</tool_call>           ← correct
    - <tool_call>{json}\n{...}</tool_call>            ← hallucinated
    - <tool_call>{tool_response}\n{...}</tool_call>   ← hallucinated
    """
    # Extract everything between <tool_call> tags
    block_match = re.search(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL)
    if not block_match:
        return None

    block = block_match.group(1).strip()

    # Remove common garbage prefixes the model hallucinates
    block = re.sub(r"^\{json\}\s*", "", block)
    block = re.sub(r"^\{tool_response\}\s*", "", block)
    block = re.sub(r"^```json\s*", "", block)
    block = re.sub(r"```\s*$", "", block)
    block = block.strip()

    # Try to find a valid JSON object inside the block
    # (model sometimes puts result data after the call JSON)
    json_matches = re.findall(r"\{[^{}]*(?:\{[^{}]*\}[^{}]*)?\}", block, re.DOTALL)

    for candidate in json_matches:
        try:
            parsed = json.loads(candidate)
            # Valid tool call must have 'name' key
            if "name" in parsed:
                return parsed
        except json.JSONDecodeError:
            continue

    # Last resort: try to extract name and location with regex
    name_match = re.search(r'"name"\s*:\s*"([^"]+)"', block)
    loc_match  = re.search(r'"location"\s*:\s*"([^"]+)"', block)
    unit_match = re.search(r'"unit"\s*:\s*"([^"]+)"', block)

    if name_match and loc_match:
        return {
            "name": name_match.group(1),
            "arguments": {
                "location": loc_match.group(1),
                "unit": unit_match.group(1) if unit_match else "celsius",
            }
        }

    return None


def run_agent_v2(user_message: str):
    """Fixed agentic loop with robust parser and strict prompt."""
    messages = [
        {"role": "system", "content": STRICT_SYSTEM},
        {"role": "user",   "content": user_message},
    ]

    print(f"\n{'='*55}")
    print(f"  User: {user_message}")
    print(f"{'='*55}")

    # ── Turn 1 ────────────────────────────────────────────────────────────
    inp = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(inp, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = 80,   # Short — tool call shouldn't be long
            temperature        = 0.05, # Nearly greedy — less hallucination
            do_sample          = True,
            repetition_penalty = 1.2,
            pad_token_id       = tokenizer.eos_token_id,
        )

    turn1 = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print(f"\n🤖 Model (turn 1): {turn1}")

    # ── Parse with robust parser ──────────────────────────────────────────
    tool_call = robust_parse_tool_call(turn1)

    if not tool_call:
        print(f"\n⚠️  No valid tool call found. Raw output above.")
        print(f"💬 Direct answer:\n{turn1}")
        return

    fn_name = tool_call.get("name")
    fn_args = tool_call.get("arguments") or tool_call.get("parameters", {})

    print(f"\n🔧 Parsed Tool Call: {fn_name}({fn_args})")

    if fn_name not in TOOLS:
        print(f"❌ Unknown tool: {fn_name}")
        return

    # ── Execute real tool ─────────────────────────────────────────────────
    tool_result = TOOLS[fn_name](**fn_args)
    print(f"📦 Real Result: {json.dumps(tool_result, indent=2)}")

    # ── Turn 2: inject result, get natural answer ─────────────────────────
    messages.append({"role": "assistant", "content": turn1})
    messages.append({
        "role": "user",
        "content": (
            f"<tool_result>\n{json.dumps(tool_result)}\n</tool_result>\n"
            "Now answer the user's original question using ONLY the data above. "
            "Be friendly and concise."
        )
    })

    inp2 = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs2 = tokenizer(inp2, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out2 = model.generate(
            **inputs2,
            max_new_tokens     = 150,
            temperature        = 0.7,
            do_sample          = True,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
        )

    final = tokenizer.decode(
        out2[0][inputs2["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print(f"\n💬 Final Answer:\n{final}")
    print(f"\n{'='*55}\n")
    return final


# ── Run tests ─────────────────────────────────────────────────────────────
run_agent_v2("What's the weather like in Dhaka right now?")
run_agent_v2("Tell me the weather in London in fahrenheit.")
run_agent_v2("Is it hot in Tokyo today?")

## 🔷 Step 6 — Save the Model

You have **3 options** below. Pick the one that fits your workflow.

In [17]:
# ────────────────────────────────────────────────────────────────────────────
# OPTION A — Save LoRA adapters only (small, ~50MB)
# Use this if you want to continue training in Notebook 2 from these adapters
# ────────────────────────────────────────────────────────────────────────────
SAVE_PATH = "qwen2.5-0.5B-tool-calling-lora"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ LoRA adapters saved to: ./{SAVE_PATH}")

✅ LoRA adapters saved to: ./qwen2.5-0.5B-tool-calling-lora


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# OPTION B — Save merged model in float16 (full weights, ~1GB)
# Use this to get a standalone model for deployment or Notebook 2 continued training
# ────────────────────────────────────────────────────────────────────────────
MERGED_PATH = "qwen2.5-0.5B-tool-calling-merged"

model.save_pretrained_merged(
    MERGED_PATH,
    tokenizer,
    save_method = "merged_16bit",  # Options: "merged_16bit", "merged_4bit", "lora"
)
print(f"✅ Merged model saved to: ./{MERGED_PATH}")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# OPTION C — Push to HuggingFace Hub (requires HF token)
# Get your token from https://huggingface.co/settings/tokens
# ────────────────────────────────────────────────────────────────────────────
PUSH_TO_HUB = True  # Set True to enable

if PUSH_TO_HUB:
    HF_USERNAME  = "naimulislam864"   # ← CHANGE THIS
    REPO_NAME    = "qwen2.5-0.5B-tool-calling"

    from huggingface_hub import login
    login()  # Will prompt for token

    model.push_to_hub_merged(
        f"{HF_USERNAME}/{REPO_NAME}",
        tokenizer,
        save_method = "merged_16bit",
    )
    print(f"✅ Model pushed to: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")
else:
    print("ℹ️  Hub upload skipped. Set PUSH_TO_HUB = True to enable.")

## ✅ Done!

Your model is now fine-tuned for **tool calling**.

**Next steps:**
- ➡️ Open **Notebook 2** to continue training for natural conversation
- Load from `qwen2.5-0.5B-tool-calling-merged` in Notebook 2 for sequential training
- Or use the LoRA adapter path if you want to keep them separate

| Saved Path | Size | Use case |
|---|---|---|
| `qwen2.5-0.5B-tool-calling-lora` | ~50 MB | Further training (Notebook 2) |
| `qwen2.5-0.5B-tool-calling-merged` | ~1 GB | Deployment / Notebook 2 base |